In [1]:
import time
import pandas as pd
from Bio import Entrez, SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
 
from load_sample_fusions import load_all_sample_fusions
 
Entrez.email = "ojeromebright@gmail.com"
 


ModuleNotFoundError: No module named 'load_sample_fusions'

In [2]:

genetic_code = {
 
    "ATA":"I","ATC":"I","ATT":"I","ATG":"M",
    "ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K",
    "AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L",
    "CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q",
    "CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V",
    "GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E",
    "GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S",
    "TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*","TGA":"*"
 
}
 
STANDARD_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")
 
WINDOW_SIZE = 9   # 8-11mer range from the literature; adjust as needed
 
 
def translate(dna):
 
    protein = ""
 
    for i in range(0, len(dna) - 2, 3):
 
        codon = dna[i:i + 3]
        amino_acid = genetic_code.get(codon, "X")
 
        if amino_acid == "*":
            break
 
        protein += amino_acid
 
    return protein
 
 


# NCBI CDS RETRIEVAL -- cached, rate-limited, error-handled

In [4]:
# cds_cache avoids re-fetching the same gene twice across many
# samples that share it. rate limiting keeps us under NCBI's
# request limits. try/except means one bad gene never kills the
# whole run -- it's logged and skipped instead.


In [5]:

def get_cds_candidates(gene, cds_cache, max_candidates=5):
 
    # Returns a LIST of candidate CDS sequences for this gene (different
    # RefSeq transcript isoforms), instead of committing to just the
    # first one found. This matters because a fusion caller's breakpoint
    # fragment might not exist in whichever transcript NCBI happens to
    # return first -- trying several candidates recovers real fusions
    # that would otherwise be wrongly logged as "fragments not found".
 
    if gene in cds_cache:
        return cds_cache[gene]
 
    print(f"Retrieving CDS candidates for {gene}")
 
    time.sleep(0.34)   # ~3 requests/second, within NCBI's no-API-key limit
 
    candidates = []
 
    try:
        search = Entrez.esearch(
            db="nucleotide",
            term=f"{gene}[Gene] AND Homo sapiens[Organism] AND biomol_mrna[PROP] AND srcdb_refseq[PROP]"
        )
        result = Entrez.read(search)
 
        if len(result["IdList"]) == 0:
            print("No RefSeq mRNA found for", gene)
            cds_cache[gene] = candidates
            return candidates
 
        for nucleotide_id in result["IdList"]:
 
            if len(candidates) >= max_candidates:
                break
 
            time.sleep(0.34)
 
            handle = Entrez.efetch(
                db="nucleotide", id=nucleotide_id, rettype="gb", retmode="text"
            )
            record = SeqIO.read(handle, "genbank")
 
            for feature in record.features:
                if feature.type == "CDS":
                    cds = feature.extract(record.seq)
                    print("  candidate:", record.id, "for", gene)
                    candidates.append(str(cds).upper())
                    break
 
        cds_cache[gene] = candidates
        return candidates
 
    except Exception as error:
        print("NCBI error for", gene, ":", error)
        cds_cache[gene] = candidates
        return candidates
 
 


In [ ]:
# PEPTIDE PROPERTY CALCULATION